# ARC-AGI-2 — Test-Time Bayesian Program Induction (offline submission)

This notebook produces a `submission.json` for the ARC Prize 2026 ARC-AGI-2
competition using the **offline** solver from the `arc_solver` package:
fitted analytic proposals + MDL-ordered enumerative program search, with a
pass@2-aware posterior-mode selector. It runs **fully offline** (no internet,
no API calls) inside the Kaggle sandbox.

**Method in one line:** for each task we induce a short program (over a compact
grid DSL) that reproduces *all* demonstration pairs, weight admissible programs
by a Minimum-Description-Length prior, and emit the posterior-mode output as
`attempt_1` and the most probable *disagreeing* output as `attempt_2`.

To run on Kaggle, attach this repository's `src/` directory as a dataset
(e.g. `arc-solver-src`) so `arc_solver` is importable. The cell below searches
common locations automatically.

In [ ]:
import os, sys, glob

# Make the arc_solver package importable from any of the usual locations.
CANDIDATES = [
    '/kaggle/input/arc-solver-src/src',
    '/kaggle/input/arc-solver/src',
    os.path.join(os.getcwd(), 'src'),
    os.path.join(os.path.dirname(os.getcwd()), 'src'),
]
for c in CANDIDATES:
    if os.path.isdir(os.path.join(c, 'arc_solver')):
        sys.path.insert(0, c)
        print('arc_solver from', c)
        break
else:
    raise RuntimeError('arc_solver not found; attach src/ as a dataset')

from arc_solver.cli.run_eval import build_engine
from arc_solver.io.loader import load_task
from arc_solver.io.submission import SubmissionWriter

In [ ]:
# Locate the evaluation tasks. The Kaggle competition mounts them under
# /kaggle/input/...; fall back to the local public data for a dry run.
DATA_CANDIDATES = [
    '/kaggle/input/arc-prize-2026/evaluation',
    '/kaggle/input/arc-prize-2026-arc-agi-2/evaluation',
    'data/arc_agi_2/evaluation',
    '../data/arc_agi_2/evaluation',
]
DATA_DIR = next((d for d in DATA_CANDIDATES if os.path.isdir(d)), None)
paths = sorted(glob.glob(os.path.join(DATA_DIR, '*.json'))) if DATA_DIR else []
print('data dir:', DATA_DIR, '| tasks:', len(paths))

In [ ]:
import time

# Offline engine: fitted proposals first, then bounded enumerative search.
engine = build_engine(max_seconds=10.0, max_programs=20000, max_depth=4)
writer = SubmissionWriter()

TOTAL_BUDGET_S = 11 * 3600   # wall-clock guard, well under the 12h limit
start = time.monotonic()
solved = 0

for i, path in enumerate(paths):
    task = load_task(path)
    if time.monotonic() - start > TOTAL_BUDGET_S:
        # Out of time: emit a safe identity/mirror fallback so no task is missing.
        attempts = []
        for p in task.test:
            g = p.input
            mirror = tuple(tuple(reversed(r)) for r in g)
            attempts.append((g, mirror if mirror != g else g))
        writer.add(task.task_id, attempts)
        continue
    result = engine.solve(task)
    solved += int(result.solved_train)
    writer.add(task.task_id, list(result.attempts))
    if (i + 1) % 20 == 0:
        print(f'{i + 1}/{len(paths)} tasks, {solved} with a program, '
              f'{time.monotonic() - start:.0f}s')

writer.write('submission.json')
print('wrote submission.json:', len(writer), 'tasks,', solved, 'with an admissible program')

The full method, theory, and ablations are described in the accompanying
write-up (`WRITEUP.md`). The offline path above is the reproducible core; an
optional LLM-amortised proposer (`arc_solver.search.llm_proposer`) raises
accuracy when an API key is available, but is disabled here because the Kaggle
sandbox has no network access.